### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="lending_club",
    dataset_year="2018",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://zenodo.org/records/11295916",
    download_description="""
It seems the official download portal from https://www.lendingclub.com is gone. So we can only utilize artifacts from the past, which seem to go up to 2018.
Old script and source: https://github.com/nateGeorge/preprocess_lending_club_data

From all the artifacts we found online, this seems to be the newest: https://www.kaggle.com/datasets/wordsforthewise/lending-club/data
And the following paper (https://arxiv.org/abs/2401.16458) curated this dataset https://zenodo.org/records/11295916 for tabular-text learning.

Other artifacts:
- https://www.kaggle.com/datasets/adarshsng/lending-club-loan-data-csv
- https://www.kaggle.com/datasets/imsparsh/lending-club-loan-dataset-2007-2011

wget https://zenodo.org/records/11295916/files/LC_loans_granting_model_dataset.csv?download=1
mkdir -p local-data-warehouse/lending_club && mv LC_loans_granting_model_dataset.csv?download=1 local-data-warehouse/lending_club
kaggle datasets download wordsforthewise/lending-club -f accepted_2007_to_2018Q4.csv.gz && mv accepted_2007_to_2018Q4.csv.gz local-data-warehouse/lending_club/
""",
    # References
    academic_reference_bibtex=r"""@article{sanz2025credit,
  title={Credit Risk Meets Large Language Models: Building a Risk Indicator from Loan Descriptions in P2P Lending},
  author={Sanz-Guerrero, Mario and Arroyo, Javier},
  journal={Inteligencia Artificial},
  volume={28},
  number={75},
  pages={220--247},
  year={2025}
}
""",
    academic_reference_bibtex_key="sanz2025credit",
    license="CC0: Public Domain", # On Kaggle, likely not true TOS from the website.
    data_tags=["Non-IID", "Temporal", "Spatial"],
    curation_comments="""
We start withe data from Sanz-Guerrero et al. (2025). This data is already well preprocessed and curated for the task and used for tabular-text learning. However, we noticed a lot variables from the original data are missing that can be added to the task without invalidating the task or introducing data leakage. Thus, we also merge new features from the original data into the version from Sanz-Guerrero et al. (2025).

- This is a datasets where the description is very often empty or a standard phrase. So the model needs to be able to handle cases with a lot of missing data in the text modality, and also cases where the text is rarely informative. This is a common case in real-world tabular-text datasets, and it is important to have it represented in our benchmark.
- We reverse the ordinal encoding of the target.
- We reverse the name change of "revenue" back to "annual_inc".
- We also create a sec_app_fico_n like the fico_n from anz-Guerrero et al. (2025).
- We drop rows where "application_type" is missing as these rows have consistent missing values across features and likely represent some data loading artifact.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Default",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Default",
    time_on="issue_d",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

original_df = pd.read_csv(dataset_mold.path /  "accepted_2007_to_2018Q4.csv.gz")
df = pd.read_csv(dataset_mold.path / "LC_loans_granting_model_dataset.csv?download=1")
print("Loaded data shape:", df.shape)
print("Original data shape:", original_df.shape)

# 0 for fully paid loans and a 1 for defaulted loans.
df["Default"] = df["Default"].map({0: "Fully Paid", 1: "Defaulted"})

df = df.rename(columns={"revenue": "annual_inc"})

# Transform     "sec_app_fico_range_low"    "sec_app_fico_range_high", to sec_app_fico_n
original_df["sec_app_fico_n"] = (original_df["sec_app_fico_range_low"] + original_df["sec_app_fico_range_high"]) / 2

new_features_from_original = original_df[[
    "id",
    # known when the loan is granted, so it can be used as a feature without data leakage.
    #   This, however, would create a different task, in that we want to train a model that the company uses after their own risk assessment but before granting the loan.
    #   Given that these values and grades are often determined by an internal model of the company, they might also introduce an unknown bias.
    #   We stick to treating the task as a pre-assessment modelling. An alternative version could also function as a "stacking" model that takes the output of the company's internal model as an input.
    #   The variables for this task are:
    # "term",
    # "int_rate",
    # "installment",
    # "grade",
    # "sub_grade",
    # "verification_status",
    # Known at the start / from data source even without the company assessing the risk, so it can be used no matter the use case.
    "emp_title",
    "disbursement_method",
    "acc_now_delinq",
    "acc_open_past_24mths",
    "all_util",
    "sec_app_fico_n",
    # annual_inc	annual_inc_joint -> were already merged by prior work
    # fico_range_high fico_range_low -> transformed before into avg fico
    "application_type", "avg_cur_bal", "bc_open_to_buy", "bc_util", "chargeoff_within_12_mths", "collections_12_mths_ex_med", "delinq_2yrs", "delinq_amnt", "earliest_cr_line", "il_util", "inq_fi", "inq_last_12m", "inq_last_6mths", "max_bal_bc", "mo_sin_old_il_acct", "mo_sin_old_rev_tl_op", "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl", "mort_acc", "mths_since_last_delinq", "mths_since_last_major_derog", "mths_since_last_record", "mths_since_rcnt_il", "mths_since_recent_bc", "mths_since_recent_bc_dlq", "mths_since_recent_inq", "mths_since_recent_revol_delinq", "num_accts_ever_120_pd", "num_actv_bc_tl", "num_actv_rev_tl", "num_bc_sats", "num_bc_tl", "num_il_tl", "num_op_rev_tl", "num_rev_accts", "num_rev_tl_bal_gt_0", "num_sats", "num_tl_120dpd_2m", "num_tl_30dpd", "num_tl_90g_dpd_24m", "num_tl_op_past_12m", "open_acc", "open_acc_6m", "open_il_12m", "open_il_24m", "open_act_il", "open_rv_12m", "open_rv_24m", "pct_tl_nvr_dlq", "percent_bc_gt_75", "pub_rec", "pub_rec_bankruptcies", "revol_bal", "revol_util", "tax_liens", "tot_coll_amt", "tot_cur_bal", "tot_hi_cred_lim", "total_acc", "total_bal_ex_mort", "total_bal_il", "total_bc_limit", "total_cu_tl", "total_il_high_credit_limit", "total_rev_hi_lim", "revol_bal_joint", "sec_app_earliest_cr_line", "sec_app_inq_last_6mths", "sec_app_mort_acc", "sec_app_open_acc", "sec_app_revol_util", "sec_app_open_act_il", "sec_app_num_rev_accts", "sec_app_chargeoff_within_12_mths", "sec_app_collections_12_mths_ex_med", "sec_app_mths_since_last_major_derog", "sec_app_fico_range_low", "sec_app_fico_range_high", # dti dti_joint -> already merged by prior work via max
]]

# Merge new features into the curated version of the dataset.
df = df.merge(new_features_from_original, on="id", how="left")
del new_features_from_original, original_df

as_cat_type = [
    # There are only two options on this website. This is ordinal technically, so we could treat it as a number too.
    "home_ownership_n",
    "application_type",
    "disbursement_method",
    "emp_length",
    "Default",
    "experience_c",
    "purpose",
]
as_string_type = [
    "emp_title",
    "addr_state",
    "zip_code",
    "title",
    "desc",
]

df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y")
df["sec_app_earliest_cr_line"] = pd.to_datetime(df["sec_app_earliest_cr_line"], format="%b-%Y")
df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y")

for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")


df[as_cat_type] = df[as_cat_type].astype("category")

df = df[~df["application_type"].isna()]
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df.drop(columns=[
    "id",  # meaningless identifier
    "experience_c", # constant after preprocessing
])

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
    duplicate_column_check=False, # skip
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
time_df = df.copy()
time_df["year_month"] = time_df[task_mold.time_on].dt.to_period("Y")

# 1) Total number of samples per month
monthly_totals = (
    time_df
    .groupby("year_month")
    .size()
    .rename("total_samples")
)

# Optional: sort by month and convert PeriodIndex to timestamp (month start)
result = monthly_totals.sort_index()
result.index = result.index.to_timestamp()
result

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata

# Sort by time
df = df.sort_values(by=task_mold.time_on).reset_index(drop=True)

splits = {}

test_years = [
    "2016",
    "2017",
    "2018"
]
for i, month in enumerate(test_years):
    ref_date  = pd.Timestamp(month)
    train_index = df[
        df[task_mold.time_on] < ref_date
    ].index
    test_index = df[
        (df[task_mold.time_on].dt.year == ref_date.year)
    ].index
    splits[i] = {
        0: (train_index.tolist(), test_index.tolist())
    }

for s in splits:
    train_index, test_index = splits[s][0]
    print(f"Split {s}: Train size: {len(train_index)}, Test size: {len(test_index)}")
    assert df[task_mold.time_on].iloc[train_index].max() < df[task_mold.time_on].iloc[test_index].min()

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="""We try to create splits that simulate a model deployed to solve the task.

The official data is updated monthly but has not enough data per month to create large enough test splits. We opt for simulating a model that is refit every year to obtain a robust test set instead. This introduces the unrealistic downside of data shift across a year that would not exist in a real-world model. We create 3 test splits (2016, 2017, 2018). For each test split, we use all data before the test month as training data.
""",
    splits=splits,
    time_horizon=1,
    time_horizon_unit="years",
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)